In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
# Install Dependencies
%pip install -qq pymupdf easyocr langchain-core langchain-text-splitters sentence-transformers chromadb langchain-groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.

In [2]:
import os
import shutil
import uuid
from pathlib import Path
import fitz  # PyMuPDF
import easyocr
import numpy as np
import PIL.Image
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# --- Paths Configuration ---
DATA_DIR = "/content/drive/MyDrive/QuesGen"
DB_DIR = "/content/drive/MyDrive/QuesGen/db"

print("Loading models...")
reader = easyocr.Reader(['en'])
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def extract_scanned_pdfs(src_dir):
    all_docs = []
    # Added ** to recursively search all subfolders (DBMS, DSA, etc.)
    files = list(Path(src_dir).glob("**/*.pdf"))
    print(f"Found {len(files)} PDFs in {src_dir}. Starting extraction...")

    for file in files:
        print(f"Reading: {file.name}")
        try:
            pdf_doc = fitz.open(file)
            for page_num in range(len(pdf_doc)):
                page = pdf_doc[page_num]
                pix = page.get_pixmap(dpi=150)
                img = PIL.Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                text = "\n".join(reader.readtext(np.array(img), detail=0))

                if text.strip():
                    all_docs.append(Document(
                        page_content=text,
                        metadata={"source_file": file.name, "page": page_num + 1}
                    ))
        except Exception as e:
            print(f"Error on {file.name}: {e}")
    return all_docs

def create_vector_store(documents, db_path, collection_name="pdf_documents"):
    if os.path.exists(db_path):
        print(f"Clearing old database at {db_path}...")
        shutil.rmtree(db_path)

    os.makedirs(db_path, exist_ok=True)
    client = chromadb.PersistentClient(path=db_path, settings=Settings(allow_reset=True))
    collection = client.get_or_create_collection(name=collection_name)

    print("Splitting text into chunks...")
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len)
    chunks = splitter.split_documents(documents)

    print(f"Embedding {len(chunks)} chunks and saving to ChromaDB...")
    contents = [chunk.page_content for chunk in chunks]
    embeddings = embedder.encode(contents).tolist()

    ids = [f"doc_{uuid.uuid4().hex[:8]}_{i}" for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=embeddings, metadatas=metas, documents=contents)
    print(f"✅ Database created successfully at {db_path}!")

# Execute extraction and database creation
docs = extract_scanned_pdfs(DATA_DIR)
if docs:
    create_vector_store(docs, DB_DIR)
else:
    print("No PDFs found to process.")

Loading models...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Found 27 PDFs in /content/drive/MyDrive/QuesGen. Starting extraction...
Reading: btech-it-3-sem-data-structure-and-algorithms-2011.pdf
Reading: btech-it-3-sem-data-structures-and-algorithm-2011.pdf
Reading: btech-it-3-sem-data-structure-and-algorithms-2013.pdf
Reading: btech-it-3-sem-data-structure-and-algorithms-pcc-csbs301-2023.pdf
Reading: btech-it-3-sem-data-structure-and-algorithms-v2-2013.pdf
Reading: btech-it-4-sem-object-oriented-programming-and-uml-2015.pdf
Reading: btech-it-4-sem-object-oriented-programming-and-uml-2016.pdf
Reading: btech-it-4-sem-object-oriented-programming-and-uml-2012.pdf
Reading: btech-it-4-sem-object-oriented-programming-and-uml-2014.pdf
Reading: btech-it-4-sem-object-oriented-programming-and-uml-2013.pdf
Reading: btech-it-6-sem-database-management-system-2017.pdf
Reading: btech-it-6-sem-database-management-system-2016.pdf
Reading: btech-it-6-sem-database-management-system-2015.pdf
Reading: btech-it-6-sem-database-management-system-2019.pdf
Reading: btec

In [3]:
# Zip the database folder
!zip -r /content/quesgen_db.zip /content/drive/MyDrive/QuesGen/db

# Download it directly to your computer
from google.colab import files
files.download('/content/quesgen_db.zip')

  adding: content/drive/MyDrive/QuesGen/db/ (stored 0%)
  adding: content/drive/MyDrive/QuesGen/db/chroma.sqlite3 (deflated 54%)
  adding: content/drive/MyDrive/QuesGen/db/ae9a7ee7-70d9-4d61-8405-b19c78802997/ (stored 0%)
  adding: content/drive/MyDrive/QuesGen/db/ae9a7ee7-70d9-4d61-8405-b19c78802997/header.bin (deflated 63%)
  adding: content/drive/MyDrive/QuesGen/db/ae9a7ee7-70d9-4d61-8405-b19c78802997/data_level0.bin (deflated 100%)
  adding: content/drive/MyDrive/QuesGen/db/ae9a7ee7-70d9-4d61-8405-b19c78802997/length.bin (deflated 65%)
  adding: content/drive/MyDrive/QuesGen/db/ae9a7ee7-70d9-4d61-8405-b19c78802997/link_lists.bin (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>